# FaceGen — Conditional Diffusion Training on CelebA
**Modèle de diffusion conditionnel** : génère des visages **nets** en choisissant 40 attributs physiques.

Workflow :
1. GPU check
2. Installer les dépendances
3. Kaggle — download dataset
4. Monter Drive (checkpoints)
5. Cloner le repo GitHub
6. Login wandb
7. Entraîner
8. Générer des visages (+ effet de la guidance)

## 1 · GPU check

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

## 2 · Install dependencies

In [ ]:
# Colab ships with PyTorch — we only need Lightning, wandb, kaggle
!pip install -q pytorch-lightning wandb kaggle

## 3 · Kaggle — credentials & download dataset

In [ ]:
# Kaggle credentials via Colab Secrets (icône 🔑 dans le panneau gauche)
# Ajoute deux secrets : KAGGLE_USERNAME et KAGGLE_KEY
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")

In [ ]:
import os, zipfile

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Download (~1.7 GB)
!kaggle datasets download -d jessicali9530/celeba-dataset -p {DATA_DIR}

# Extract main zip
print("Extracting…")
with zipfile.ZipFile(f"{DATA_DIR}/celeba-dataset.zip") as z:
    z.extractall(DATA_DIR)
os.remove(f"{DATA_DIR}/celeba-dataset.zip")

# Les images sont dans un zip imbriqué sur ce dataset Kaggle
nested = f"{DATA_DIR}/img_align_celeba.zip"
if os.path.exists(nested):
    with zipfile.ZipFile(nested) as z:
        z.extractall(DATA_DIR)
    os.remove(nested)

# Sanity check
img_dir = f"{DATA_DIR}/img_align_celeba/img_align_celeba"
n = len(os.listdir(img_dir))
print(f"Images : {n:,}")
assert n == 202_599, f"Structure inattendue — vérifie {DATA_DIR}"

## 4 · Google Drive — checkpoints uniquement

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = "/content/drive/MyDrive/FaceGen/checkpoints_diffusion"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoints → {CKPT_DIR}")

## 5 · Clone repo

In [ ]:
REPO_URL = "https://github.com/JulesV19/face-gen.git"
REPO_DIR = "/content/face-gen"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

## 6 · Weights & Biases login

In [ ]:
import wandb
wandb.login()  # paste your API key when prompted

## 7 · Training

Diffusion = plus lourd que le VAE (~3-4× le temps/epoch). Les premiers visages
corrects apparaissent vers epoch 15-30, vraiment nets vers 50+. Des échantillons
sont loggés dans wandb tous les `sample_every_n_epochs` (défaut 5).

In [ ]:
# L4 / A100 · bf16-mixed natif (Ada / Ampere)
# UNet ~37M params, batch_size=128 @ 64×64
!python train_diffusion.py \
    --data_dir {DATA_DIR} \
    --batch_size 128 \
    --max_epochs 100 \
    --lr 2e-4 \
    --timesteps 1000 \
    --schedule cosine \
    --cond_drop_prob 0.1 \
    --ema_decay 0.9999 \
    --guidance_scale 3.0 \
    --ddim_steps 50 \
    --sample_every_n_epochs 5 \
    --precision bf16-mixed \
    --num_workers 4 \
    --ckpt_dir {CKPT_DIR} \
    --wandb_project facegen-diffusion

## 8 · Inference
### 8a · Génération — choisir les attributs

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import torch
from generate_diffusion import load_model
from generate import build_condition
from utils import show_grid, list_attrs

# ── EDIT: path to your best checkpoint ───────────────────────────────────────
CKPT = "/content/drive/MyDrive/FaceGen/checkpoints_diffusion/best.ckpt"

device = "cuda"
unet, diffusion, cfg = load_model(CKPT, device)

print("All available attributes:")
list_attrs()

In [ ]:
# ── Customise here ────────────────────────────────────────────────────────────
# 1.0 = attribute present, 0.0 = absent
wanted_attrs = {
    "Smiling":          1.0,
    "Blond_Hair":       1.0,
    "Young":            1.0,
    "Wearing_Lipstick": 1.0,
    "High_Cheekbones":  1.0,
}

c = build_condition(wanted_attrs, cfg.num_attrs, default=0.0, device=device)
attrs = c.unsqueeze(0).expand(16, -1).contiguous()

imgs = diffusion.ddim_sample(
    unet, attrs, guidance_scale=3.0, ddim_steps=50, img_size=cfg.img_size
)
show_grid(imgs, nrow=8, title="Generated faces (diffusion, guidance=3)")

### 8b · Effet de la classifier-free guidance

Même bruit initial, on fait varier `guidance_scale`. Plus haut = attributs plus
marqués mais moins de diversité / risque de sur-saturation. 3 est un bon défaut.

In [ ]:
attrs = build_condition(
    {"Smiling": 1.0, "Eyeglasses": 1.0}, cfg.num_attrs, default=0.0, device=device
).unsqueeze(0).expand(8, -1).contiguous()

rows = []
for g in [1.0, 2.0, 3.0, 5.0]:
    torch.manual_seed(0)  # même bruit initial → comparaison propre
    rows.append(
        diffusion.ddim_sample(unet, attrs, guidance_scale=g, ddim_steps=50, img_size=cfg.img_size)
    )

show_grid(torch.cat(rows), nrow=8, title="guidance = 1 / 2 / 3 / 5  (haut → bas)", figsize=(16, 9))